In [3]:
import sys
import os

# Add the parent directory to the path so the package is importable
sys.path.append(os.path.abspath(".."))

from llm_data_quality_assistant.pipeline import Pipeline
from llm_data_quality_assistant.enums import Models, CorruptionTypes
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
import numpy as np
import jupyter_helper_functions
import string
import time

load_dotenv()

True

In [4]:
gold_standard = pd.read_csv(
    "../datasets/parker_datasets/allergen/allergen_cleaned_gold_first1000.csv"
)
corrupted_dataset = jupyter_helper_functions.load_dataset(
    "../datasets/parker_datasets/allergen_custom/allergen_custom_corruption.csv"
)
corrupted_dataset = corrupted_dataset.fillna(0)
corrupted_dataset = corrupted_dataset.astype(int)

corrupted_dataset = Pipeline.generate_corrupted_datasets(
    gold_standard,
    cell_corruption_types=[
        CorruptionTypes.CellCorruptionTypes.OUTLIER,
    ],
    row_corruption_types=[
    ],
    columns_to_exclude=["code"],
    severity=0.4,
    output_size=1,
    inplace=False
)
corrupted_dataset = corrupted_dataset[0]

# Calculate percentage of cells that differ from gold_standard
# Make copies to avoid modifying the originals
gold_standard_copy = gold_standard.copy()
corrupted_dataset_copy = corrupted_dataset.copy()
gold_standard_copy = gold_standard_copy.drop(columns=["code"])
corrupted_dataset_copy = corrupted_dataset_copy.drop(columns=["code"])

# Calculate percentage of cells that differ from gold_standard_copy
total_cells_copy = gold_standard_copy.size
diff_cells_copy = (gold_standard_copy != corrupted_dataset_copy).sum().sum()
percentage_diff_copy = (diff_cells_copy / total_cells_copy) * 100
print(f"Percentage of differing cells (copy): {percentage_diff_copy:.2f}%")

output = Pipeline.standardize_datasets("code", gold_standard=gold_standard, corrupted_dataset = corrupted_dataset)
gold_standard = output["gold_standard"]
corrupted_dataset = output["corrupted_dataset"]

# Duplicate and append the DataFrame 5 times
# corrupted_versions = 5
# gold_standard_extended = pd.concat([gold_standard.copy() for _ in range(corrupted_versions)], ignore_index=True)
# gold_standard_extended = pd.concat([group for _, group in gold_standard_extended.groupby("dicom_uid")], ignore_index=True)

Percentage of differing cells (copy): 40.01%


In [5]:
# corrupted_dataset = Pipeline.generate_corrupted_datasets(
#     dataset=gold_standard,
#     cell_corruption_types=[CorruptionTypes.CellCorruptionTypes.NULL],
#     row_corruption_types=[],
#     columns_to_exclude=["code"],
#     inplace=True,
#     severity=0.15,
#     output_size=1
# )
# corrupted_dataset = corrupted_dataset[0]
# jupyter_helper_functions.save_dataframe_csv(corrupted_dataset, "../datasets/parker_datasets/allergen_custom/allergen_custom_corruption.csv")
# raise ValueError("dlfsjk")

In [6]:
rpm = 0
model_name = Models.OpenAIModels.GPT_4_1_MINI
context_rows = 0
file_name = jupyter_helper_functions.sanitize_filename(f"{model_name.value}_{context_rows}_rows_context")
primary_key = "code"

additional_context = f"""
{corrupted_dataset.sample(n=context_rows).to_string(index=False)}
"""

# Merge/clean with LLM
merged_df, time_taken = jupyter_helper_functions.merge_with_llm_timed(
    dataset=corrupted_dataset,
    primary_key=primary_key,
    model=model_name,
    rpm=rpm,
    additional_prompt=additional_context
)


Merging groups with LLM: 100%|██████████| 103/103 [06:16<00:00,  3.66s/it]


In [7]:
jupyter_helper_functions.save_dataframe_csv(merged_df, f"../analysis/repairs/allergen_custom/{file_name}_repair.csv")

In [8]:
import json


# Evaluate results
jupyter_helper_functions.standardize_and_evaluate(
    gold_standard=gold_standard,
    merged_df=merged_df,
    corrupt_dataset=corrupted_dataset,
    primary_key=primary_key,
    time_delta=time_taken,
    results_dir=f"../analysis/results/allergen_custom/",
    file_name=file_name,
)

{'accuracy': 0.3934350439204808,
 'column_names': ['code',
                  'nuts',
                  'almondnuts',
                  'brazil_nuts',
                  'macadamia_nuts',
                  'hazelnut',
                  'pistachio',
                  'walnut',
                  'cashew',
                  'celery',
                  'crustaceans',
                  'eggs',
                  'fish',
                  'gluten',
                  'lupin',
                  'milk',
                  'molluscs',
                  'mustard',
                  'peanut',
                  'sesame',
                  'soy',
                  'sulfite'],
 'f1_score': 0.12240802675585284,
 'false_negative': 1548,
 'false_negative_rate': 0.8942807625649913,
 'false_positive': 1076,
 'false_positive_rate': 0.4146435452793834,
 'num_columns': 22,
 'num_rows': 206,
 'precision': 0.1453534551231136,
 'recall': 0.10571923743500866,
 'time_taken': 376.741503238678,
 'true_negative': 1519,
